In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from src.data.preprocess import load_snapshots
from src.models.gcn import GCN
from src.models.graphsage import GraphSAGE
from src.training.trainer import train_epoch, evaluate

## Load Data

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

snapshots = load_snapshots(raw_dir='../data/raw')

train_snaps = [s for s in snapshots if s.timestep <= 34]
val_snaps   = [s for s in snapshots if 35 <= s.timestep <= 39]
test_snaps  = [s for s in snapshots if s.timestep >= 40]

print(f'Train: {len(train_snaps)} | Val: {len(val_snaps)} | Test: {len(test_snaps)}')

# pos_weight to handle class imbalance
n_licit   = sum((s.y[s.train_mask] == 0).sum().item() for s in train_snaps)
n_illicit = sum((s.y[s.train_mask] == 1).sum().item() for s in train_snaps)
pos_weight = torch.tensor([n_licit / n_illicit]).to(DEVICE)
print(f'pos_weight: {pos_weight.item():.2f}  (licit={n_licit:,}, illicit={n_illicit:,})')

Device: cpu
Train: 34 | Val: 5 | Test: 10
pos_weight: 7.63  (licit=26,432, illicit=3,462)


## Config

In [3]:
IN_CHANNELS = 164   # 164 features — timestep (feat_1) excluded
HIDDEN      = 128
EPOCHS      = 100   # matches teammate (faisal_Project_DL_7643.ipynb)
LR          = 1e-3
WD          = 1e-4

## Training helper

In [4]:
import os
os.makedirs('../models', exist_ok=True)

def run_experiment(model, name, epochs=EPOCHS, lr=LR, wd=WD):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-5
    )

    history = []
    best_f1, best_weights = 0, None

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_snaps, optimizer, pos_weight, DEVICE)
        val  = evaluate(model, val_snaps, DEVICE)
        history.append({'epoch': epoch, 'loss': loss, **val})

        scheduler.step(val['f1'])

        if val['f1'] > best_f1:
            best_f1      = val['f1']
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'[{name}] epoch {epoch:3d} | loss {loss:.4f} | '
                  f'val F1 {val["f1"]:.4f} | val PR-AUC {val["pr_auc"]:.4f} | lr {cur_lr:.2e}')

    # Restore and save best checkpoint
    model.load_state_dict(best_weights)
    torch.save(best_weights, f'../models/{name.lower()}_best.pt')
    print(f'  Saved best checkpoint → models/{name.lower()}_best.pt')

    test = evaluate(model, test_snaps, DEVICE)
    print(f'\n[{name}] best val F1={best_f1:.4f}')
    print(f'[{name}] TEST  F1={test["f1"]:.4f}  ROC-AUC={test["roc_auc"]:.4f}  PR-AUC={test["pr_auc"]:.4f}\n')
    return pd.DataFrame(history), test, model

## Run GCN

In [5]:
gcn_history, gcn_test, gcn_model = run_experiment(
    GCN(in_channels=IN_CHANNELS, hidden_channels=HIDDEN),
    'GCN'
)

[GCN] epoch  10 | loss 0.3738 | val F1 0.4338 | val PR-AUC 0.7533 | lr 1.00e-03
[GCN] epoch  20 | loss 0.2782 | val F1 0.4954 | val PR-AUC 0.7902 | lr 1.00e-03
[GCN] epoch  30 | loss 0.2150 | val F1 0.5612 | val PR-AUC 0.8439 | lr 1.00e-03
[GCN] epoch  40 | loss 0.1800 | val F1 0.5552 | val PR-AUC 0.8003 | lr 1.00e-03
[GCN] epoch  50 | loss 0.1506 | val F1 0.6542 | val PR-AUC 0.7868 | lr 1.00e-03
[GCN] epoch  60 | loss 0.1466 | val F1 0.6904 | val PR-AUC 0.8150 | lr 1.00e-03
[GCN] epoch  70 | loss 0.1352 | val F1 0.6667 | val PR-AUC 0.8084 | lr 1.00e-03
[GCN] epoch  80 | loss 0.1284 | val F1 0.6410 | val PR-AUC 0.8319 | lr 1.00e-03
[GCN] epoch  90 | loss 0.0952 | val F1 0.6285 | val PR-AUC 0.8256 | lr 5.00e-04
[GCN] epoch 100 | loss 0.0875 | val F1 0.6764 | val PR-AUC 0.8074 | lr 5.00e-04
  Saved best checkpoint → models/gcn_best.pt

[GCN] best val F1=0.7095
[GCN] TEST  F1=0.4145  ROC-AUC=0.8515  PR-AUC=0.3919



## Run GraphSAGE

In [6]:
sage_history, sage_test, sage_model = run_experiment(
    GraphSAGE(in_channels=IN_CHANNELS, hidden_channels=HIDDEN),
    'GraphSAGE'
)

[GraphSAGE] epoch  10 | loss 0.2302 | val F1 0.4673 | val PR-AUC 0.8837 | lr 1.00e-03
[GraphSAGE] epoch  20 | loss 0.1260 | val F1 0.6215 | val PR-AUC 0.8668 | lr 1.00e-03
[GraphSAGE] epoch  30 | loss 0.0891 | val F1 0.6700 | val PR-AUC 0.8551 | lr 1.00e-03
[GraphSAGE] epoch  40 | loss 0.0909 | val F1 0.6299 | val PR-AUC 0.8724 | lr 1.00e-03
[GraphSAGE] epoch  50 | loss 0.0651 | val F1 0.7808 | val PR-AUC 0.8536 | lr 1.00e-03
[GraphSAGE] epoch  60 | loss 0.0521 | val F1 0.7432 | val PR-AUC 0.8307 | lr 1.00e-03
[GraphSAGE] epoch  70 | loss 0.0397 | val F1 0.7730 | val PR-AUC 0.8731 | lr 1.00e-03
[GraphSAGE] epoch  80 | loss 0.0416 | val F1 0.7758 | val PR-AUC 0.8677 | lr 1.00e-03
[GraphSAGE] epoch  90 | loss 0.0233 | val F1 0.7898 | val PR-AUC 0.8712 | lr 5.00e-04
[GraphSAGE] epoch 100 | loss 0.0237 | val F1 0.8096 | val PR-AUC 0.8516 | lr 5.00e-04
  Saved best checkpoint → models/graphsage_best.pt

[GraphSAGE] best val F1=0.8201
[GraphSAGE] TEST  F1=0.4921  ROC-AUC=0.8503  PR-AUC=0.426

## Run TGN

TGN maintains a **global memory vector** that evolves across timesteps via a GRU.
At each timestep every node is conditioned on this memory before Graph Attention aggregation.

Key difference from EvolveGCN:
- EvolveGCN evolves the GCN **weight matrix** W
- TGN evolves a **memory vector** and injects it into node features

In [7]:
from src.models.tgn import TGN
from src.training.trainer import train_epoch_temporal, evaluate_temporal

TGN_LR     = 5e-4
TGN_EPOCHS = 100   # matches teammate epoch count
TGN_WD     = 1e-4

def run_tgn_experiment():
    model     = TGN(in_channels=IN_CHANNELS, memory_dim=HIDDEN, hidden_dim=HIDDEN).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=TGN_LR, weight_decay=TGN_WD)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-5
    )
    history = []
    best_pr, best_weights = 0, None

    for epoch in range(1, TGN_EPOCHS + 1):
        loss = train_epoch_temporal(model, train_snaps, optimizer, pos_weight, DEVICE)
        val  = evaluate_temporal(model, train_snaps, val_snaps, DEVICE)
        history.append({'epoch': epoch, 'loss': loss, **val})

        scheduler.step(val['pr_auc'])

        if val['pr_auc'] > best_pr:
            best_pr      = val['pr_auc']
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'[TGN] epoch {epoch:3d} | loss {loss:.4f} | '
                  f'val F1 {val["f1"]:.4f} | val PR-AUC {val["pr_auc"]:.4f} | lr {cur_lr:.2e}')

    # Restore and save best checkpoint
    model.load_state_dict(best_weights)
    torch.save(best_weights, '../models/tgn_best.pt')
    print('  Saved best checkpoint → models/tgn_best.pt')

    test = evaluate_temporal(model, train_snaps + val_snaps, test_snaps, DEVICE)
    print(f'\n[TGN] best val PR-AUC = {best_pr:.4f}')
    print(f'[TGN] TEST  F1={test["f1"]:.4f}  ROC-AUC={test["roc_auc"]:.4f}  PR-AUC={test["pr_auc"]:.4f}')
    return pd.DataFrame(history), test, model

tgn_history, tgn_test, tgn_model = run_tgn_experiment()

[TGN] epoch  10 | loss 0.4523 | val F1 0.5100 | val PR-AUC 0.7210 | lr 5.00e-04
[TGN] epoch  20 | loss 0.3014 | val F1 0.6006 | val PR-AUC 0.8100 | lr 5.00e-04
[TGN] epoch  30 | loss 0.2204 | val F1 0.7116 | val PR-AUC 0.8162 | lr 5.00e-04
[TGN] epoch  40 | loss 0.1753 | val F1 0.7493 | val PR-AUC 0.8463 | lr 5.00e-04
[TGN] epoch  50 | loss 0.1474 | val F1 0.7343 | val PR-AUC 0.8470 | lr 5.00e-04
[TGN] epoch  60 | loss 0.1216 | val F1 0.7652 | val PR-AUC 0.8596 | lr 5.00e-04
[TGN] epoch  70 | loss 0.1095 | val F1 0.7743 | val PR-AUC 0.8557 | lr 5.00e-04
[TGN] epoch  80 | loss 0.0987 | val F1 0.7817 | val PR-AUC 0.8724 | lr 5.00e-04
[TGN] epoch  90 | loss 0.0777 | val F1 0.7695 | val PR-AUC 0.8655 | lr 2.50e-04
[TGN] epoch 100 | loss 0.0697 | val F1 0.7763 | val PR-AUC 0.8675 | lr 2.50e-04
  Saved best checkpoint → models/tgn_best.pt

[TGN] best val PR-AUC = 0.8745
[TGN] TEST  F1=0.5276  ROC-AUC=0.8739  PR-AUC=0.5728


---
## Results & Report Figures

Run this section **after all four models have finished training**.  
Produces five publication-ready figures saved to `../figures/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
os.makedirs('../figures', exist_ok=True)

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
    'savefig.dpi':       180,
    'savefig.bbox':      'tight',
})

COLORS = {
    'ResGCN':    '#4a90d9',
    'ResSAGE':   '#1f5fa6',
    'EvolveGCN': '#e07b39',
    'TGN':       '#c0392b',
}

# ── EvolveGCN: use teammate's results (our implementation incomplete)
# Source: faisal_Project_DL_7643.ipynb
# Teammate split: train t=1–30, test t=35–49, torch_geometric_temporal library
EVOLVE_TEAMMATE = {'f1': 0.3079, 'roc_auc': 0.7979, 'pr_auc': 0.1922}

# ── Build results DataFrame ───────────────────────────────────────────────────
results = pd.DataFrame([
    {'Model': 'ResGCN',    'Type': 'Static',   **gcn_test},
    {'Model': 'ResSAGE',   'Type': 'Static',   **sage_test},
    {'Model': 'EvolveGCN', 'Type': 'Temporal', **EVOLVE_TEAMMATE},
    {'Model': 'TGN',       'Type': 'Temporal', **tgn_test},
]).set_index('Model')

results['best_val_f1'] = [
    gcn_history['f1'].max(),
    sage_history['f1'].max(),
    float('nan'),            # teammate's val history unavailable
    tgn_history['f1'].max(),
]

# ── Print summary table ───────────────────────────────────────────────────────
disp = results[['Type', 'f1', 'roc_auc', 'pr_auc', 'best_val_f1']].copy()
disp.columns = ['Type', 'Test F1', 'ROC-AUC', 'PR-AUC', 'Best Val F1']

print("=" * 68)
print("  GraphShield — Test Results  (Elliptic, t = 40–49)")
print("  Split: train 1–34 | val 35–39 | test 40–49")
print("  † EvolveGCN result from teammate (faisal_Project_DL_7643.ipynb)")
print("=" * 68)
print(disp.round(4).to_string())
print("=" * 68)
our_results = results.drop('EvolveGCN')
print(f"  Best model by Test F1:    {our_results['f1'].idxmax()}  ({our_results['f1'].max():.4f})")
print(f"  Best model by ROC-AUC:    {our_results['roc_auc'].idxmax()}  ({our_results['roc_auc'].max():.4f})")
print(f"  Best model by PR-AUC:     {our_results['pr_auc'].idxmax()}  ({our_results['pr_auc'].max():.4f})")
print("=" * 68)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Figure 1 — F1 Comparison Bar Chart
#  Vertical bars · static/temporal shading · Weber et al. RF baseline
# ═══════════════════════════════════════════════════════════════════

models = list(results.index)   # ResGCN, ResSAGE, EvolveGCN, TGN
f1s    = results['f1'].tolist()
colors = [COLORS[m] for m in models]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(models, f1s, color=colors, edgecolor='black', linewidth=0.6, width=0.55)

# Background shading — static vs temporal
ax.axvspan(-0.5, 1.5, color='#e3f2fd', alpha=0.35, zorder=0)
ax.axvspan( 1.5, 3.5, color='#fff3e0', alpha=0.40, zorder=0)

ax.text(0.5, 0.965, 'static',   ha='center', va='top',
        transform=ax.get_xaxis_transform(), color='#1565c0', fontsize=9, style='italic')
ax.text(2.5, 0.965, 'temporal', ha='center', va='top',
        transform=ax.get_xaxis_transform(), color='#bf360c', fontsize=9, style='italic')

# Weber et al. 2019 Random Forest reference
ax.axhline(0.79, color='#2e7d32', linestyle='--', linewidth=1.4, alpha=0.75,
           label='Random Forest — Weber et al. 2019*')

for bar, v in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.015,
            f'{v:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Test illicit-class F1')
ax.set_ylim(0, 1.05)
ax.set_title(
    'Fraud detection — Elliptic Bitcoin dataset\n'
    'Temporal split: train t=1–34 | val t=35–39 | test t=40–49',
    fontsize=11,
)
ax.legend(loc='upper right', frameon=False, fontsize=9)
ax.text(0.99, -0.12,
        '*RF result from Weber et al. 2019 — evaluated under a different split; shown for reference only.',
        transform=ax.transAxes, ha='right', fontsize=8, style='italic', color='gray')

plt.tight_layout()
plt.savefig('../figures/fig1_f1_comparison.png')
plt.show()
print("Saved → figures/fig1_f1_comparison.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Figure 2 — Multi-Metric Grouped Bar Chart
#  F1 / ROC-AUC / PR-AUC shown per model side-by-side
# ═══════════════════════════════════════════════════════════════════

metric_cols   = ['f1',  'roc_auc', 'pr_auc']
metric_labels = ['F1',  'ROC-AUC', 'PR-AUC']
alphas        = [1.0,   0.68,      0.40]

x_pos   = np.arange(len(models))
bar_w   = 0.22
offsets = np.linspace(-(len(metric_cols)-1)/2, (len(metric_cols)-1)/2, len(metric_cols)) * bar_w

fig, ax = plt.subplots(figsize=(10, 5))

for j, (col, label, alpha) in enumerate(zip(metric_cols, metric_labels, alphas)):
    vals = results[col].tolist()
    xj   = x_pos + offsets[j]
    bars = ax.bar(xj, vals, width=bar_w, label=label,
                  color=colors, alpha=alpha, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels(models)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.08)
ax.set_title('Test-set performance across all metrics', fontsize=12, fontweight='bold')

ax.axvspan(-0.5, 1.5, color='#e3f2fd', alpha=0.22, zorder=0)
ax.axvspan( 1.5, 3.5, color='#fff3e0', alpha=0.28, zorder=0)
ax.text(0.5, 0.97, 'static',   ha='center', va='top',
        transform=ax.get_xaxis_transform(), color='#1565c0', fontsize=9, style='italic')
ax.text(2.5, 0.97, 'temporal', ha='center', va='top',
        transform=ax.get_xaxis_transform(), color='#bf360c', fontsize=9, style='italic')

legend_patches = [Patch(facecolor='gray', alpha=a, label=l)
                  for l, a in zip(metric_labels, alphas)]
ax.legend(handles=legend_patches, loc='upper right', frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig('../figures/fig2_multi_metric.png')
plt.show()
print("Saved → figures/fig2_multi_metric.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Figure 3 — Learning Curves
#  Validation F1, Validation PR-AUC, and Training Loss across epochs
#  Note: EvolveGCN excluded — no training history (teammate result used)
# ═══════════════════════════════════════════════════════════════════

curve_data = [
    (gcn_history,  'ResGCN',  COLORS['ResGCN'],  '-'),
    (sage_history, 'ResSAGE', COLORS['ResSAGE'], '-'),
    (tgn_history,  'TGN',     COLORS['TGN'],     '--'),
    # EvolveGCN omitted — no training history available (teammate result)
]

panels = [
    ('f1',     'Validation F1',     (0.0, 1.0)),
    ('pr_auc', 'Validation PR-AUC', (0.0, 1.0)),
    ('loss',   'Training Loss',     None),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (col, ylabel, ylim) in zip(axes, panels):
    for hist, name, color, ls in curve_data:
        if col in hist.columns:
            ax.plot(hist['epoch'], hist[col], label=name,
                    color=color, linewidth=2, linestyle=ls, alpha=0.9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontweight='bold')
    if ylim:
        ax.set_ylim(*ylim)
    ax.legend(fontsize=9, frameon=False)
    ax.grid(alpha=0.18)
    ax.text(0.03, 0.97, '— static   ╌╌ temporal',
            transform=ax.transAxes, va='top', fontsize=8, color='#666', style='italic')

plt.suptitle('Training dynamics — GraphShield models\n'
             '(EvolveGCN excluded — teammate result used)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/fig3_learning_curves.png')
plt.show()
print("Saved → figures/fig3_learning_curves.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Figure 4 — Validation → Test Generalisation Gap
#  Only models we trained ourselves are included (ResGCN, ResSAGE, TGN).
#  EvolveGCN is excluded — val history not available (teammate result).
# ═══════════════════════════════════════════════════════════════════

our_models   = ['ResGCN', 'ResSAGE', 'TGN']
our_colors   = [COLORS[m] for m in our_models]
best_val_f1s = results.loc[our_models, 'best_val_f1'].tolist()
test_f1s     = results.loc[our_models, 'f1'].tolist()
gaps         = [bv - tf for bv, tf in zip(best_val_f1s, test_f1s)]

x_pos = np.arange(len(our_models))
w     = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(x_pos - w/2, best_val_f1s, w,
       label='Best Validation F1',
       color=our_colors, alpha=0.95, edgecolor='black', linewidth=0.5)
ax.bar(x_pos + w/2, test_f1s, w,
       label='Test F1  (t = 40–49)',
       color=our_colors, alpha=0.42, edgecolor='black', linewidth=0.5, hatch='//')

for i, (bv, tf, gap) in enumerate(zip(best_val_f1s, test_f1s, gaps)):
    top = max(bv, tf) + 0.025
    ax.annotate(f'Δ = {gap:.3f}',
                xy=(x_pos[i], top), ha='center', fontsize=9.5,
                color='#333', fontweight='bold')

ax.axhline(0.79, color='#2e7d32', linestyle=':', linewidth=1.3, alpha=0.6,
           label='Weber et al. RF baseline (0.79)*')

ax.set_xticks(x_pos)
ax.set_xticklabels(our_models)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.12)
ax.set_title(
    'Validation → Test generalisation gap\n'
    'Δ reflects concept drift from law-enforcement crackdown at t ≈ 43',
    fontsize=11, fontweight='bold',
)
ax.legend(frameon=False, fontsize=9)
ax.grid(axis='y', alpha=0.18)

ax.text(0.99, -0.13,
        'EvolveGCN excluded (no val history — teammate result). '
        '*RF baseline from Weber et al. 2019; different split.',
        transform=ax.transAxes, ha='right', fontsize=8, style='italic', color='gray')

plt.tight_layout()
plt.savefig('../figures/fig4_generalisation_gap.png')
plt.show()
print("Saved → figures/fig4_generalisation_gap.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Figure 5 — GraphShield vs. Teammate  (F1 head-to-head)
#
#  Teammate results (faisal_Project_DL_7643.ipynb):
#    GCN       F1=0.4335   test t≥35, checkpoint by test F1
#    GraphSAGE F1=0.6966   same conditions
#    EvolveGCN F1=0.3079   torch_geometric_temporal library
#
#  Our setup: test t≥40, checkpoint by best validation metric only.
# ═══════════════════════════════════════════════════════════════════

TEAMMATE_F1 = {'ResGCN': 0.4335, 'ResSAGE': 0.6966, 'EvolveGCN': 0.3079}

shared  = ['ResGCN', 'ResSAGE', 'EvolveGCN']
ours_f1 = [results.loc[m, 'f1'] for m in shared]
team_f1 = [TEAMMATE_F1[m] for m in shared]

x_pos = np.arange(len(shared))
w     = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x_pos - w/2, ours_f1, w,
       color=[COLORS[m] for m in shared],
       edgecolor='black', linewidth=0.6, alpha=0.95,
       label='GraphShield (ours) — test t=40–49, val checkpoint')

ax.bar(x_pos + w/2, team_f1, w,
       color=[COLORS[m] for m in shared],
       edgecolor='black', linewidth=0.6, alpha=0.38, hatch='xx',
       label='Teammate — test t=35–49, test-set checkpoint')

for i, (o, t) in enumerate(zip(ours_f1, team_f1)):
    ax.text(x_pos[i] - w/2, o + 0.012, f'{o:.3f}',
            ha='center', fontsize=10, fontweight='bold')
    ax.text(x_pos[i] + w/2, t + 0.012, f'{t:.3f}',
            ha='center', fontsize=10, color='#555')

# TGN — our model only
tgn_f1 = results.loc['TGN', 'f1']
ax.bar([3.0], [tgn_f1], w,
       color=COLORS['TGN'], edgecolor='black', linewidth=0.6, alpha=0.95)
ax.text(3.0, tgn_f1 + 0.012, f'{tgn_f1:.3f}',
        ha='center', fontsize=10, fontweight='bold', color=COLORS['TGN'])
ax.annotate('TGN\n(ours only)',
            xy=(3.0, tgn_f1 + 0.05), xytext=(3.0, tgn_f1 + 0.18),
            ha='center', fontsize=9, color=COLORS['TGN'], fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLORS['TGN'], lw=1.3),
            bbox=dict(boxstyle='round,pad=0.3', fc='#fff3f3', ec=COLORS['TGN'], alpha=0.8))

ax.set_xticks(list(x_pos) + [3.0])
ax.set_xticklabels(shared + ['TGN'])
ax.set_ylabel('Test illicit-class F1')
ax.set_ylim(0, 0.97)
ax.set_title(
    'GraphShield vs. Teammate — Test F1 Comparison\n'
    'Our evaluation uses a harder test split with no test-set leakage',
    fontsize=11, fontweight='bold',
)
ax.legend(frameon=False, fontsize=9, loc='upper left')
ax.grid(axis='y', alpha=0.18)

ax.text(0.99, -0.13,
        'Teammate checkpoints by test F1; test split starts at t=35. '
        'Our checkpoint by best validation metric; test split starts at t=40.',
        transform=ax.transAxes, ha='right', fontsize=8, style='italic', color='gray')

plt.tight_layout()
plt.savefig('../figures/fig5_vs_teammate.png')
plt.show()
print("Saved → figures/fig5_vs_teammate.png")

print()
print("=" * 55)
print("  All figures saved to ../figures/")
print("  fig1_f1_comparison.png      — main F1 bar chart")
print("  fig2_multi_metric.png       — F1 / ROC-AUC / PR-AUC")
print("  fig3_learning_curves.png    — val F1, PR-AUC, loss")
print("  fig4_generalisation_gap.png — val→test concept drift")
print("  fig5_vs_teammate.png        — head-to-head comparison")
print("=" * 55)